# Session 1 — Returns, and Why Prices Lie

**Hands-on hour (55 min).** Fill in the `TODO`s. Don't copy-paste from ChatGPT/Claude — the friction is the learning.

**Objectives:** compute simple & log returns · annualize correctly · *see* fat tails and volatility clustering in real data.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats

# SPY (index) + three stocks from different sectors
tickers = ["SPY", "JPM", "AAPL", "XOM"]
px = yf.download(tickers, start="2016-01-01", auto_adjust=True)["Close"]
px = px.dropna()
px.tail()

[*********************100%***********************]  4 of 4 completed


Ticker,AAPL,JPM,SPY,XOM
Date,,,,
2026-06-30,289.359985,325.862000,746.770020,136.720001
2026-07-01,294.380005,332.571808,745.760010,136.279999
2026-07-02,308.630005,332.970001,744.780029,137.089996
2026-07-06,312.660004,337.720001,751.280029,136.440002
2026-07-07,310.660004,339.220001,747.710022,141.690002


## Part A — Simple vs log returns (10 min)

Simple: $R_t = \frac{P_t}{P_{t-1}} - 1$  ·  Log: $r_t = \ln\frac{P_t}{P_{t-1}}$

Compute both. Then check: how different are they day-to-day? On the single worst day?

In [30]:
simple_ret = px.pct_change().dropna()
log_ret = np.log(px / px.shift(1)).dropna()  # TODO: use np.log(px / px.shift(1)), drop NaNs

# TODO: max absolute difference between the two, per ticker
ab_diff_max = np.abs(simple_ret - log_ret).max()

print(ab_diff_max)
# TODO: find SPY's worst day (idxmin) — compare simple vs log return on that day
worst_diff_SPY = np.abs(simple_ret['SPY'].min() - log_ret['SPY'].min())

diff_aapl_max = np.abs(simple_ret['AAPL'].max() - log_ret['AAPL'].max())
diff_aapl_min = np.abs(simple_ret['AAPL'].min() - log_ret['AAPL'].min())

print(diff_aapl_max)
print(diff_aapl_min)
print(np.abs(simple_ret - log_ret).mean())
# Expected insight: nearly identical for small moves, diverge on big ones

Ticker
AAPL    0.010671
JPM     0.014505
SPY     0.006463
XOM     0.008143
dtype: float64
0.010671062953977378
0.009061094261949176
Ticker
AAPL    0.000166
JPM     0.000149
SPY     0.000063
XOM     0.000155
dtype: float64


## Part B — Annualization (10 min)

Convention: 252 trading days. Mean scales by 252, volatility by $\sqrt{252}$ (variance is additive under independence, so vol grows with the *square root* of time).

In [38]:
ann_ret = log_ret.mean() * 252 # TODO: log_ret.mean() * 252
ann_vol = log_ret.std() * np.sqrt(252)  # TODO: log_ret.std() * np.sqrt(252)

summary = pd.DataFrame({"ann_return": ann_ret, "ann_vol": ann_vol})
# TODO: add a naive 'sharpe_0' column = ann_return / ann_vol (rf=0 for now; proper Sharpe in Session 3)
summary['sharpe_0'] = ann_ret / ann_vol
summary['CAGR'] = np.exp(ann_ret) - 1
summary

,ann_return,ann_vol,sharpe_0,CAGR
Ticker,,,,
AAPL,0.245589,0.289071,0.849581,0.278374
JPM,0.186291,0.273819,0.680343,0.204773
SPY,0.141686,0.178737,0.792708,0.152215
XOM,0.100735,0.279731,0.360114,0.105984


## Part C — Fat tails: see them yourself (20 min)

If returns were normal, a 4σ day would happen ~ once per 63 years. Count how many SPY has had since 2016.

In [ ]:
r = log_ret["SPY"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram vs fitted normal
axes[0].hist(r, bins=100, density=True, alpha=0.6)
x = np.linspace(r.min(), r.max(), 500)
# TODO: overlay stats.norm.pdf(x, r.mean(), r.std()) on axes[0]
axes[0].set_title("SPY daily log returns vs fitted Normal")

# QQ-plot
stats.probplot(r, dist="norm", plot=axes[1])
plt.tight_layout()

# TODO: excess kurtosis with stats.kurtosis(r) — Normal = 0. What do you get?
# TODO: count days where |r| > 4 * r.std(). Normal theory predicts ~0.02 such days in this sample.
# TODO (optional): repeat histogram with MONTHLY returns — closer to normal? (aggregational Gaussianity)

## Part D — Volatility clustering (15 min)

Returns themselves are ~unpredictable, but their *magnitude* is not. Two plots prove it.

In [ ]:
# 1) Rolling 21-day annualized vol
roll_vol = None  # TODO: r.rolling(21).std() * np.sqrt(252)
# TODO: plot it. Label the spikes you recognize (2018 Q4, COVID 2020, 2022...)

# 2) Autocorrelation: returns vs |returns|, lags 1..20
lags = range(1, 21)
ac_r   = [r.autocorr(l) for l in lags]
ac_abs = None  # TODO: same but for r.abs()

# TODO: bar-plot both side by side.
# Expected: ac_r ≈ 0 everywhere; ac_abs clearly positive for many lags.
# That asymmetry is THE core stylized fact — sit with it for a minute.

## Review (15 min — closed book)

Write answers in a new markdown cell below **from memory**, then check against your notes:

1. Why do quants prefer log returns? (two reasons)
2. Why is assuming normality of returns dangerous? Cite a number from your own Part C output.
3. What is volatility clustering, and which plot proved it?
4. Why does volatility annualize with √252 and not 252?

Then tick Session 1 in `Quant_Finance_20h_Plan.md` and paste your four answers to Claude for grading.